# 01 · Data preparation

**Goal.** Build a clean, fully encoded dataset from the original TCGA export and prove that it is identical to the encoded file published by UCI.

Why rebuild it instead of using the UCI file directly? The UCI file (`TCGA_InfoWithGrade.csv`) drops the patient identifier (`Case_ID`) and the histological diagnosis (`Primary_Diagnosis`). We need the identifier to trace patients and the diagnosis to (i) demonstrate the leakage problem and (ii) prepare the multiclass task planned for January. Rebuilding from `TCGA_GBM_LGG_Mutations_all.csv` and checking the result value by value against UCI gives both: the extra columns and the guarantee that our cleaning is exactly the published one.

**Outputs** (all in `results/`):
- `tables/data_dictionary.tex`, `tables/data_cleaning.tex`, `tables/data_uci_check.tex`, `tables/data_feature_sets.tex`
- `logs/01_data_preparation.json`
- `data/processed/glioma_clean.csv` (regenerated each run, not versioned)

In [ ]:
# Make the project root importable, whatever folder the notebook is run from
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Project root (folder containing src/config.py) not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import config, data, reporting

pd.set_option("display.max_columns", 40)
print("Project root:", ROOT)

## 1. Load the raw files

Two files are provided. `Mutations_all` keeps the original text labels; `InfoWithGrade` is the version UCI encoded as integers after removing incomplete patients. The raw files are only read, never modified.

In [ ]:
raw = data.load_raw_mutations()
uci = data.load_uci_encoded()

print(f"Original file : {raw.shape[0]} patients x {raw.shape[1]} columns")
print(f"UCI file      : {uci.shape[0]} patients x {uci.shape[1]} columns")
print("Columns only in the original file:", sorted(set(raw.columns) - set(uci.columns)))
display(raw.head())

## 2. Data quality

Missing values are not stored as empty cells: the TCGA export writes `--`, and race can also be `not reported`. A plain `isna()` would find nothing, so we count these markers explicitly. We also check that no patient appears twice.

In [ ]:
quality = data.count_invalid_markers(raw)
display(quality)
print("Duplicated Case_ID:", raw[config.ID_COL].duplicated().sum())
print("Values found in the gene columns:", sorted(set(raw[config.GENE_COLS].stack())))

## 3. Leakage columns

`Project` (TCGA-LGG / TCGA-GBM) is the TCGA sub-project each patient comes from. If it coincides with `Grade`, it is the target under another name and must be dropped. `Primary_Diagnosis` is kept only as metadata: the cross-tabulation shows that the diagnosis determines the grade, so it can never be a predictor of `Grade` (and vice versa).

In [ ]:
project_is_grade = data.check_leakage_columns(raw)
print("Project identical to Grade:", project_is_grade)
display(pd.crosstab(raw[config.MULTICLASS_TARGET], raw["Grade"], margins=True))

## 4. Remove incomplete patients

A patient is removed if a clinical field needed by the project is missing. Each patient is counted under the first reason that applies, so the counts add up. The removed patients are listed below to check that they are not concentrated in one class.

In [ ]:
clean, removal_table, removed = data.clean_and_encode(raw)
display(removal_table)
print("Grade of the removed patients:", removed["Grade"].value_counts().to_dict())
display(removed[[config.ID_COL, "Grade", "Gender", "Age_at_diagnosis", "Race", "Reason"]])

## 5. Encoding

Age is converted from `"X years Y days"` to years (`X + Y/365`, rounded to 2 decimals). All categorical variables are encoded as integers with the same codes as UCI (see the data dictionary below).

`Race` is **nominal**: the codes 0-3 have no order, so treating them as a number would be wrong for linear models and SVMs. It is kept with the UCI codes here and one-hot encoded inside each model pipeline (stages 4 and 5).

In [ ]:
display(clean.head())
print(clean.dtypes.value_counts())
dictionary = data.data_dictionary()
display(dictionary)

## 6. Check against the UCI file

The rebuilt dataset must coincide with `TCGA_InfoWithGrade.csv` in the number of patients, their order and every value. If any check fails, the cleaning is not the published one and the notebook stops.

In [ ]:
checks = data.verify_against_uci(clean, uci)
uci_check = pd.DataFrame({"Check": list(checks), "Passed": list(checks.values())})
display(uci_check)
assert all(checks.values()), "The rebuilt dataset differs from the UCI file"
print("The rebuilt dataset is identical to the UCI file.")

## 7. Final dataset and feature sets

Two facts to keep in mind for the next stages:
- **Rare categories.** Some race codes have very few patients (see counts). In cross-validation a category may be absent from a training fold; the one-hot encoder will be set to ignore unknown categories.
- **Sparse mutations.** Several genes are mutated in fewer than 5% of patients. They are kept (they may still be informative), but the EDA will quantify their individual relevance.

In [ ]:
print("Grade:", clean["Grade"].map(config.CLASS_LABELS).value_counts().to_dict())
print("Race :", clean["Race"].value_counts().sort_index().to_dict())
prevalence = clean[config.GENE_COLS].mean().sort_values()
print("Genes mutated in < 5% of patients:", prevalence[prevalence < 0.05].round(3).to_dict())

feature_sets = pd.DataFrame([
    {"Task": "Classification", "Target": "Grade",
     "Predictors": ", ".join(data.feature_columns("classification")[:3]) + " + 20 genes",
     "Number of predictors": len(data.feature_columns("classification"))},
    {"Task": "Regression", "Target": "Age_at_diagnosis",
     "Predictors": ", ".join(data.feature_columns("regression")[:2]) + " + 20 genes",
     "Number of predictors": len(data.feature_columns("regression"))},
    {"Task": "Unsupervised", "Target": "None",
     "Predictors": "20 genes", "Number of predictors": len(data.feature_columns("unsupervised"))},
])
display(feature_sets)

## 8. Save results

In [ ]:
processed_path = data.save_processed(clean)

reporting.save_table_tex(dictionary, "data_dictionary", column_format="p{3.2cm}p{2.4cm}p{4.6cm}p{4.2cm}")
reporting.save_table_tex(removal_table, "data_cleaning", column_format="lrr")
reporting.save_table_tex(uci_check, "data_uci_check", column_format="ll")
reporting.save_table_tex(feature_sets, "data_feature_sets", column_format="llp{6cm}r")

summary = {
    "n_raw": len(raw),
    "n_clean": len(clean),
    "n_removed": len(removed),
    "removal_steps": removal_table,
    "removed_by_grade": removed["Grade"].value_counts().to_dict(),
    "removed_case_ids": removed[config.ID_COL].tolist(),
    "duplicated_case_ids": int(raw[config.ID_COL].duplicated().sum()),
    "invalid_markers_per_column": quality.to_dict(orient="index"),
    "project_identical_to_grade": project_is_grade,
    "diagnosis_vs_grade": pd.crosstab(raw[config.MULTICLASS_TARGET], raw["Grade"]).to_dict(orient="index"),
    "uci_checks": checks,
    "identical_to_uci": all(checks.values()),
    "grade_counts": clean["Grade"].map(config.CLASS_LABELS).value_counts().to_dict(),
    "grade_gbm_proportion": float(clean["Grade"].mean()),
    "race_counts": clean["Race"].value_counts().sort_index().to_dict(),
    "gender_counts": clean["Gender"].value_counts().sort_index().to_dict(),
    "age_summary": clean["Age_at_diagnosis"].describe().round(2).to_dict(),
    "gene_prevalence": prevalence.round(4).to_dict(),
    "feature_sets": {t: data.feature_columns(t) for t in ("classification", "regression", "unsupervised")},
    "processed_file": str(processed_path.relative_to(config.ROOT)),
}
log_path = reporting.save_json(summary, "01_data_preparation")
print("Saved:", processed_path.relative_to(config.ROOT), "and", log_path.relative_to(config.ROOT))
print("Tables:", sorted(p.name for p in config.TABLES.glob("data_*.tex")))